In [ ]:
# ==============================================================================
# SADECE WIKIPEDIA İLE BAĞIMSIZ SORU ÜRETİM MOTORU (KALDIĞI YERDEN DEVAM)
# ==============================================================================

print("⏳ Kütüphaneler kontrol ediliyor...")
!pip install -q transformers accelerate bitsandbytes pandas tqdm PyMuPDF

import os
import json
import random
import pandas as pd
from tqdm import tqdm
import torch
import gc
from transformers import pipeline, BitsAndBytesConfig
import re
from google.colab import drive

# 1. Drive Bağlantısı
drive.mount('/content/drive', force_remount=True)

ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
WIKI_KLASOR = f'{ANA_KLASOR}/kaynaklar/wikipedia_ham'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Önce Modeli Yükle (Premium GPU'nun gücünü hissedeceksin)
print("\n🤖 Qwen2.5-7B modeli yükleniyor...")
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    model_kwargs={"quantization_config": quant_config},
    device_map="auto"
)
print("✅ Model başarıyla GPU'ya yerleşti!")

# 3. SADECE Wikipedia Metinlerini Okuma
print("\n📄 Sadece Wikipedia metinleri okunuyor...")
tum_metinler = []

if os.path.exists(WIKI_KLASOR):
    for dosya in os.listdir(WIKI_KLASOR):
        if dosya.endswith('.txt'):
            with open(os.path.join(WIKI_KLASOR, dosya), 'r', encoding='utf-8') as f:
                tum_metinler.append(f.read())

tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 800
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), parca_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 200]

random.seed(42)
random.shuffle(parcalar)
TOPLAM_PARCA = len(parcalar)

del tum_metinler
del tum_ham_metin
gc.collect()

# 4. KALDIĞI YERDEN DEVAM ETME MANTIĞI
HEDEF_JSON = f"{DATASET_KLASOR}/wiki_soru_cevap_yedek.json"
HEDEF_CSV = f"{DATASET_KLASOR}/wiki_soru_cevap_final.csv"

uretilen_veriler = []
baslangic_indeksi = 0

if os.path.exists(HEDEF_JSON):
    try:
        with open(HEDEF_JSON, 'r', encoding='utf-8') as f:
            uretilen_veriler = json.load(f)
        baslangic_indeksi = len(uretilen_veriler)
        print(f"🔄 Harika! Önceki veriler bulundu. Üretime {baslangic_indeksi}. sorudan DEVAM EDİLİYOR...")
    except:
        pass

kalan_parcalar = parcalar[baslangic_indeksi:]
print(f"\n🚀 SIFIR ATIF VE BAĞIMSIZ SORU ÜRETİMİ başlıyor...")

def guvenli_turkce_mi(metin):
    yasakli_kelimeler = ['the', 'is', 'are', 'what', 'how', 'why', 'and', 'to', 'of', 'in', 'that', 'this', 'it']
    yasakli_kaliplar = ['bu metinde', 'bu yazıda', 'yukarıdaki metne', 'bu süreçte', 'bu belgede', 'metne göre', 'metin', 'yazar', 'bu parça', 'bu anlatımda']

    metin_kucuk = metin.lower()
    for kelime in yasakli_kelimeler:
        if f" {kelime} " in metin_kucuk: return False
    for kalip in yasakli_kaliplar:
        if kalip in metin_kucuk: return False
    if re.search(r'[\u4e00-\u9fff]', metin): return False
    return True

for metin in tqdm(kalan_parcalar, desc="Wiki Soruları Tamamlanıyor", initial=baslangic_indeksi, total=TOPLAM_PARCA):
    prompt = f"""<|im_start|>system
Sen bir tarih uzmanısın. Görevin, sana verilen metindeki tarihi gerçekleri kullanarak, SANKİ O METİN HİÇ YOKMUŞ GİBİ bağımsız, net ve doğal bir Soru-Cevap çifti üretmektir.
Kurallar:
1. "Bu metinde", "Bu yazıda", "Yukarıdaki metne göre" gibi ifadelere KESİNLİKLE yer verme.
2. Zamir kullanma, doğrudan isim ver.
3. Sadece saf Türkçe kullan.
Format:
Soru: [Bağımsız ve net sorunuz]
Cevap: [Metne atıf yapmayan akıcı cevabınız]<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=250, temperature=0.3, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        if "Soru:" in cikti and "Cevap:" in cikti:
            soru = cikti.split("Cevap:")[0].replace("Soru:", "").strip()
            cevap = cikti.split("Cevap:")[1].strip()

            if len(soru) > 10 and len(cevap) > 30 and guvenli_turkce_mi(cikti):
                uretilen_veriler.append({"instruction": soru, "input": "", "output": cevap})

        if len(uretilen_veriler) % 10 == 0:
            with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

    except Exception:
        continue

if uretilen_veriler:
    with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
        json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
    df = pd.DataFrame(uretilen_veriler)
    df.to_csv(HEDEF_CSV, index=False, encoding='utf-8')
    print("\n==================================================")
    print(f"🎉 WIKIPEDIA İŞLEMİ TAMAMLANDI! Toplam {len(uretilen_veriler)} Soru Hazır.")
    print("==================================================")

In [ ]:
!pip install -U bitsandbytes>=0.46.1 transformers accelerate

In [ ]:
!pip install -q -U PyMuPDF

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - "BÖL VE YÖNET" STRATEJİSİ (AŞAMA 1)
# ==============================================================================

import os
import json
import random
import pandas as pd
from tqdm import tqdm
import fitz  # PyMuPDF
import torch
import gc
from transformers import pipeline, BitsAndBytesConfig
import re
from google.colab import drive

# 1. ADIM: DRIVE BAĞLANTISI
print("⏳ Google Drive'a bağlanılıyor...")
drive.mount('/content/drive', force_remount=True)

ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. ADIM: MODEL KONTROLÜ
print("\n🤖 Qwen2.5-7B modeli kontrol ediliyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-7B-Instruct",
        model_kwargs={"quantization_config": quant_config},
        device_map="auto"
    )
print("✅ Model hazır! (A100 GPU gücü devrede)")

# 3. ADIM: SADECE 1 KLASÖRÜ OKUMA (Böl ve Yönet)
# ŞU AN SADECE 'atamdergi.gov.tr' KLASÖRÜNÜ İŞLİYORUZ
HEDEF_KLASOR = 'atamdergi.gov.tr'
print(f"\n📄 SADECE '{HEDEF_KLASOR}' klasörü okunuyor...")
tum_metinler = []
klasor_yolu = os.path.join(KAYNAKLAR_KLASORU, HEDEF_KLASOR)

if os.path.exists(klasor_yolu):
    dosyalar = [f for f in os.listdir(klasor_yolu) if f.endswith('.pdf')]
    for dosya in dosyalar:
        try:
            doc = fitz.open(os.path.join(klasor_yolu, dosya))
            tum_metinler.append("".join([sayfa.get_text() for sayfa in doc]))
            doc.close()
        except Exception:
            pass

print("\n✂️ Metinler mikroskobik parçalara bölünüyor...")
tum_ham_metin = "\n".join(tum_metinler)

# Metni 600 harflik bloklara böl
parca_boyutu = 600
adim_boyutu = 400
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 200]

random.seed(42)
random.shuffle(parcalar)
TOPLAM_PARCA = len(parcalar)
print(f"✅ Lokma Küçüldü! Sadece bu klasörden {TOPLAM_PARCA} adet bilgi bloğu elde edildi!\n")

del tum_metinler
del tum_ham_metin
gc.collect()

# 4. ADIM: SORU ÜRETİMİ
HEDEF_JSON = f"{DATASET_KLASOR}/akademik_10k_dataset.json"
HEDEF_CSV = f"{DATASET_KLASOR}/akademik_10k_dataset.csv"

uretilen_veriler = []
baslangic_indeksi = 0

if os.path.exists(HEDEF_JSON):
    try:
        with open(HEDEF_JSON, 'r', encoding='utf-8') as f:
            uretilen_veriler = json.load(f)
        baslangic_indeksi = len(uretilen_veriler)
        print(f"🔄 Önceki veriler (varsa) korundu. Üretime devam ediliyor...")
    except:
        pass

kalan_parcalar = parcalar[baslangic_indeksi:]
print(f"\n🚀 {HEDEF_KLASOR} İÇİN SORU ÜRETİMİ BAŞLIYOR...")

def guvenli_turkce_mi(metin):
    yasakli_kelimeler = ['the', 'is', 'are', 'what', 'how', 'why', 'and', 'to', 'of', 'in', 'that', 'this', 'it']
    yasakli_kaliplar = ['bu metinde', 'bu yazıda', 'yukarıdaki metne', 'bu süreçte', 'bu belgede', 'metne göre', 'metin', 'yazar', 'bu parça', 'bu anlatımda', 'bu makalede', 'yazara göre']

    metin_kucuk = metin.lower()
    for kelime in yasakli_kelimeler:
        if f" {kelime} " in metin_kucuk: return False
    for kalip in yasakli_kaliplar:
        if kalip in metin_kucuk: return False
    if re.search(r'[\u4e00-\u9fff]', metin): return False
    return True

for metin in tqdm(kalan_parcalar, desc=f"{HEDEF_KLASOR} Soruları", initial=baslangic_indeksi, total=TOPLAM_PARCA):
    prompt = f"""<|im_start|>system
Sen bir tarih uzmanısın. Görevin, sana verilen metindeki tarihi gerçekleri kullanarak, SANKİ O METİN HİÇ YOKMUŞ GİBİ bağımsız, net ve doğal bir Soru-Cevap çifti üretmektir.

KURALLAR:
1. KESİNLİKLE "Bu metinde", "Bu yazıda", "Yukarıdaki metne göre" deme.
2. Zamir kullanma (Örn: "Bu antlaşma" yerine doğrudan ismini ver).
3. Cevap verirken "Metinde belirtildiği gibi" deme.
4. Sadece Türkçe kullan.

Format:
Soru: [Bağımsız ve net sorunuz]
Cevap: [Metne atıf yapmayan akıcı cevabınız]<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=250, temperature=0.3, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        if "Soru:" in cikti and "Cevap:" in cikti:
            soru = cikti.split("Cevap:")[0].replace("Soru:", "").strip()
            cevap = cikti.split("Cevap:")[1].strip()

            if len(soru) > 10 and len(cevap) > 30 and guvenli_turkce_mi(cikti):
                uretilen_veriler.append({
                    "instruction": soru,
                    "input": "",
                    "output": cevap
                })

        if len(uretilen_veriler) % 20 == 0:
            with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

    except Exception:
        continue

if uretilen_veriler:
    with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
        json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
    df = pd.DataFrame(uretilen_veriler)
    df.to_csv(HEDEF_CSV, index=False, encoding='utf-8')
    print("\n==================================================")
    print(f"🎉 {HEDEF_KLASOR} TAMAMLANDI! Toplam {len(uretilen_veriler)} Soru Hazır.")
    print("==================================================")

In [ ]:
import os
klasor_yolu = '/content/drive/MyDrive/turk_tarihi_chatbot/kaynaklar/atamdergi.gov.tr'

if os.path.exists(klasor_yolu):
    dosyalar = os.listdir(klasor_yolu)
    print(f"✅ Klasör BULUNDU! İçinde {len(dosyalar)} adet dosya var.")
    print("Dosya isimleri:", dosyalar)
else:
    print("❌ KLASÖR BULUNAMADI! Klasör adı veya yeri değişmiş olabilir.")

In [ ]:
import fitz
import os

# Sadece ilk dosyayı röntgenliyoruz
dosya_yolu = '/content/drive/MyDrive/turk_tarihi_chatbot/kaynaklar/atamdergi.gov.tr/Ataturk-Donemi-Turk-Dis-Politikasi.pdf'

print("🔍 PDF Röntgeni Başlıyor...")

try:
    doc = fitz.open(dosya_yolu)
    toplam_sayfa = len(doc)
    print(f"✅ Dosya başarıyla açıldı! Toplam {toplam_sayfa} sayfa bulundu.")

    # Sadece ilk 3 sayfadaki yazıları toplamaya çalışalım
    ornek_metin = ""
    for i in range(min(3, toplam_sayfa)):
        ornek_metin += doc[i].get_text()

    print(f"📄 İlk 3 sayfadan çıkarılabilen saf metin uzunluğu: {len(ornek_metin.strip())} karakter.")

    if len(ornek_metin.strip()) == 0:
        print("\n🚨 TEŞHİS: Bu PDF bir 'Tarama (Scanned) Fotoğraf' veya şifreli! İçinde dijital metin katmanı yok.")
    else:
        print("✅ TEŞHİS: Metin başarıyla okunuyor! İlk 100 harf şöyle:")
        print("-" * 30)
        print(repr(ornek_metin[:100]))
        print("-" * 30)

    doc.close()

except Exception as e:
    print(f"\n❌ KRİTİK HATA: PDF okunamadı. Hata mesajı: {e}")

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - ZIRHLI PDF OKUYUCU ("Böl ve Yönet" Aşama 1)
# ==============================================================================

import os
import json
import random
import pandas as pd
from tqdm import tqdm
import fitz  # PyMuPDF
import torch
import gc
from transformers import pipeline, BitsAndBytesConfig
import re
from google.colab import drive

print("⏳ Google Drive'a bağlanılıyor...")
drive.mount('/content/drive', force_remount=True)

ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

print("\n🤖 Qwen2.5-7B modeli kontrol ediliyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-7B-Instruct",
        model_kwargs={"quantization_config": quant_config},
        device_map="auto"
    )
print("✅ Model hazır! (A100 GPU gücü devrede)")

HEDEF_KLASOR = 'atamdergi.gov.tr'
print(f"\n📄 SADECE '{HEDEF_KLASOR}' klasörü ZIRHLI modda okunuyor...")
tum_metinler = []
klasor_yolu = os.path.join(KAYNAKLAR_KLASORU, HEDEF_KLASOR)

if os.path.exists(klasor_yolu):
    dosyalar = [f for f in os.listdir(klasor_yolu) if f.endswith('.pdf')]
    for dosya in dosyalar:
        print(f"📖 İşleniyor: {dosya}...")
        try:
            doc = fitz.open(os.path.join(klasor_yolu, dosya))
            basarili_sayfa = 0
            # Sırrımız Burada: Sayfaları tek tek ve korumalı okuyoruz
            for sayfa_no in range(len(doc)):
                try:
                    metin = doc[sayfa_no].get_text()
                    if metin.strip():
                        tum_metinler.append(metin)
                        basarili_sayfa += 1
                except Exception:
                    continue # Sadece hatalı SAYFAYI atla, kitabı çöpe atma!
            doc.close()
            print(f"   ✅ {basarili_sayfa} sayfa başarıyla kurtarıldı ve okundu.")
        except Exception as e:
            print(f"   ❌ {dosya} açılamadı: {e}")

print("\n✂️ Metinler mikroskobik parçalara bölünüyor...")
tum_ham_metin = "\n".join(tum_metinler)

# Metni 600 harflik bloklara böl
parca_boyutu = 600
adim_boyutu = 400
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 200]

random.seed(42)
random.shuffle(parcalar)
TOPLAM_PARCA = len(parcalar)
print(f"🎉 HARİKA! Sadece bu klasörden {TOPLAM_PARCA} adet bilgi bloğu söküp aldık!\n")

del tum_metinler
del tum_ham_metin
gc.collect()

if TOPLAM_PARCA > 0:
    # 4. ADIM: SORU ÜRETİMİ
    HEDEF_JSON = f"{DATASET_KLASOR}/atam_dataset.json"
    HEDEF_CSV = f"{DATASET_KLASOR}/atam_dataset.csv"

    uretilen_veriler = []
    baslangic_indeksi = 0

    if os.path.exists(HEDEF_JSON):
        try:
            with open(HEDEF_JSON, 'r', encoding='utf-8') as f:
                uretilen_veriler = json.load(f)
            baslangic_indeksi = len(uretilen_veriler)
            print(f"🔄 Önceki veriler korundu. {baslangic_indeksi}. sorudan devam ediliyor...")
        except:
            pass

    kalan_parcalar = parcalar[baslangic_indeksi:]
    print(f"\n🚀 {HEDEF_KLASOR} İÇİN SORU ÜRETİMİ BAŞLIYOR...")

    def guvenli_turkce_mi(metin):
        yasakli_kelimeler = ['the', 'is', 'are', 'what', 'how', 'why', 'and', 'to', 'of', 'in', 'that', 'this', 'it']
        yasakli_kaliplar = ['bu metinde', 'bu yazıda', 'yukarıdaki metne', 'bu süreçte', 'bu belgede', 'metne göre', 'metin', 'yazar', 'bu parça', 'bu anlatımda', 'bu makalede', 'yazara göre']

        metin_kucuk = metin.lower()
        for kelime in yasakli_kelimeler:
            if f" {kelime} " in metin_kucuk: return False
        for kalip in yasakli_kaliplar:
            if kalip in metin_kucuk: return False
        if re.search(r'[\u4e00-\u9fff]', metin): return False
        return True

    for metin in tqdm(kalan_parcalar, desc=f"{HEDEF_KLASOR} Soruları", initial=baslangic_indeksi, total=TOPLAM_PARCA):
        prompt = f"""<|im_start|>system
Sen bir tarih uzmanısın. Görevin, sana verilen metindeki tarihi gerçekleri kullanarak, SANKİ O METİN HİÇ YOKMUŞ GİBİ bağımsız, net ve doğal bir Soru-Cevap çifti üretmektir.

KURALLAR:
1. KESİNLİKLE "Bu metinde", "Bu yazıda", "Yukarıdaki metne göre" deme.
2. Zamir kullanma (Örn: "Bu antlaşma" yerine doğrudan ismini ver).
3. Cevap verirken "Metinde belirtildiği gibi" deme.
4. Sadece Türkçe kullan.

Format:
Soru: [Bağımsız ve net sorunuz]
Cevap: [Metne atıf yapmayan akıcı cevabınız]<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
        try:
            sonuc = generator(prompt, max_new_tokens=250, temperature=0.3, return_full_text=False)
            cikti = sonuc[0]['generated_text'].strip()

            if "Soru:" in cikti and "Cevap:" in cikti:
                soru = cikti.split("Cevap:")[0].replace("Soru:", "").strip()
                cevap = cikti.split("Cevap:")[1].strip()

                if len(soru) > 10 and len(cevap) > 30 and guvenli_turkce_mi(cikti):
                    uretilen_veriler.append({
                        "instruction": soru,
                        "input": "",
                        "output": cevap
                    })

            if len(uretilen_veriler) % 20 == 0:
                with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

        except Exception:
            continue

    if uretilen_veriler:
        with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
            json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
        df = pd.DataFrame(uretilen_veriler)
        df.to_csv(HEDEF_CSV, index=False, encoding='utf-8')
        print("\n==================================================")
        print(f"🎉 {HEDEF_KLASOR} TAMAMLANDI! Toplam {len(uretilen_veriler)} Soru Hazır.")
        print("==================================================")
else:
    print("🚨 Metin çıkarılamadığı için işlem durduruldu.")

In [ ]:
import os
import fitz

# İnceleyeceğimiz diğer klasör: 'akademik'
hedef_klasor = '/content/drive/MyDrive/turk_tarihi_chatbot/kaynaklar/akademik'

print("🔍 'akademik' Klasörü Röntgenleniyor...\n")

if os.path.exists(hedef_klasor):
    dosyalar = [f for f in os.listdir(hedef_klasor) if f.endswith('.pdf')]

    if not dosyalar:
        print("Bu klasörde PDF bulunamadı veya klasör boş.")
    else:
        print(f"✅ Toplam {len(dosyalar)} adet dosya bulundu. İşte listesi ve sayfa sayıları:\n")
        toplam_sayfa = 0
        for dosya in dosyalar:
            dosya_yolu = os.path.join(hedef_klasor, dosya)
            try:
                doc = fitz.open(dosya_yolu)
                sayfa_sayisi = len(doc)
                print(f"📄 {dosya} ---> {sayfa_sayisi} sayfa")
                toplam_sayfa += sayfa_sayisi
                doc.close()
            except Exception as e:
                print(f"❌ {dosya} okunamadı: {e}")

        print(f"\n📊 KLASÖRÜN TOPLAM SAYFA SAYISI: {toplam_sayfa}")
else:
    print("❌ 'akademik' adında bir klasör bulunamadı. Adı farklı olabilir mi?")

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - EN HIZLI VE DOĞAL ÜRETİM (MUMCU ÖZEL)
# ==============================================================================

import os, json, random, torch, gc, re
import pandas as pd
from tqdm import tqdm
import fitz
from transformers import pipeline, BitsAndBytesConfig
from google.colab import drive

# 1. Drive Bağlantısı
drive.mount('/content/drive', force_remount=True)
ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar/akademik'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Model Yükleme (A100 GPU)
print("\n🤖 Model A100 üzerine yerleşiyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                         model_kwargs={"quantization_config": quant_config}, device_map="auto")

# 3. SADECE EN KALİTELİ KAYNAK: Prof. Ahmet Mumcu
SECILI_DOSYA = 'ATATÜRK-İLKELERİ-VE-İNKILÂP-TARİHİ-I-II-Prof.-Ahmet-Mumcu.pdf'
tum_metinler = []

print(f"\n📖 En kaliteli kaynak okunuyor: {SECILI_DOSYA}")
dosya_yolu = os.path.join(KAYNAKLAR_KLASORU, SECILI_DOSYA)
if os.path.exists(dosya_yolu):
    doc = fitz.open(dosya_yolu)
    for sayfa in doc:
        metin = sayfa.get_text()
        if metin.strip(): tum_metinler.append(metin)
    doc.close()
else:
    print("🚨 Dosya bulunamadı! Lütfen yolu kontrol et.")

# 4. Parçalara Bölme
tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 700 # Biraz daha geniş alıyoruz ki bağlam kopmasın
adim_boyutu = 500
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 250]
random.shuffle(parcalar)

# 5. Soru Üretimi (Doğal Konuşma Dili Odaklı)
HEDEF_JSON = f"{DATASET_KLASOR}/ahmet_mumcu_dogal_dataset.json"
uretilen_veriler = []
if os.path.exists(HEDEF_JSON):
    with open(HEDEF_JSON, 'r', encoding='utf-8') as f: uretilen_veriler = json.load(f)

print(f"\n🚀 {len(parcalar)} parçadan DOĞAL DİLDE üretim başlıyor...")

for metin in tqdm(parcalar[len(uretilen_veriler):]):
    prompt = f"""<|im_start|>system
Sen bir tarih öğretmenisin. Görevin, verilen metinden yola çıkarak, sanki karşında bir öğrenci varmış gibi DOĞAL, AKICI ve SOHBET TADINDA bir Soru-Cevap çifti üretmektir.

KRİTİK KURALLAR:
1. Soru "Metne göre..." diye başlamasın. Doğrudan konuya gir. (Örn: "Mudanya Ateşkesi'nin en önemli sonucu neydi?")
2. Cevap akademik bir makale gibi değil, bir uzmanın anlatımı gibi akıcı olsun.
3. KESİNLİKLE "Bu metinde belirtildiği üzere" gibi ifadeler kullanma.
4. Sadece Türkçe kullan.

Format:
Soru: [Doğal soru]
Cevap: [Anlaşılır ve akıcı cevap]<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=250, temperature=0.4, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()
        if "Soru:" in cikti and "Cevap:" in cikti:
            s = cikti.split("Cevap:")[0].replace("Soru:", "").strip()
            c = cikti.split("Cevap:")[1].strip()
            # Basit Türkçe kontrolü ve atıf temizliği
            if "metin" not in cikti.lower() and len(s) > 10:
                uretilen_veriler.append({"instruction": s, "input": "", "output": c})

            if len(uretilen_veriler) % 15 == 0:
                with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
    except: continue

print(f"\n✅ İşlem tamam! {len(uretilen_veriler)} adet doğal soru Drive'ına kaydedildi.")

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - GECE MESAİSİ (SADECE TÜRK TARİHİ FİLTRELİ)
# ==============================================================================

import os, json, random, torch, gc, re
import pandas as pd
from tqdm import tqdm
import fitz
from transformers import pipeline, BitsAndBytesConfig
from google.colab import drive

# 1. Drive Bağlantısı
drive.mount('/content/drive', force_remount=True)
ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar/akademik'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Model Yükleme
print("\n🤖 Model A100 üzerine yerleşiyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                         model_kwargs={"quantization_config": quant_config}, device_map="auto")

# 3. SEÇİLİ KAYNAKLARIN OKUNMASI
SECILI_DOSYALAR = [
    'ATATÜRK-İLKELERİ-VE-İNKILÂP-TARİHİ-I-II-Prof.-Ahmet-Mumcu.pdf',
    'ataturkilkeleriveinkilaptarihii.pdf'
]

tum_metinler = []
print(f"\n📖 Gece mesaisi başlıyor. Hedef: {len(SECILI_DOSYALAR)} dev kaynak.")

for dosya in SECILI_DOSYALAR:
    dosya_yolu = os.path.join(KAYNAKLAR_KLASORU, dosya)
    if os.path.exists(dosya_yolu):
        print(f"📄 {dosya} okunuyor...")
        doc = fitz.open(dosya_yolu)
        for sayfa in doc:
            metin = sayfa.get_text()
            if metin.strip(): tum_metinler.append(metin)
        doc.close()

# 4. Parçalara Bölme
tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 650
adim_boyutu = 450
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 250]
random.seed(42)
random.shuffle(parcalar)

# 5. Üretim Ayarları (YENİ TERTEMİZ DOSYA)
HEDEF_JSON = f"{DATASET_KLASOR}/turk_tarihi_gece_dataset.json"
uretilen_veriler = []
if os.path.exists(HEDEF_JSON):
    with open(HEDEF_JSON, 'r', encoding='utf-8') as f: uretilen_veriler = json.load(f)

print(f"\n🚀 TOPLAM {len(parcalar)} parça işlenecek. SADECE TÜRK TARİHİ filtreleniyor...")

for metin in tqdm(parcalar[len(uretilen_veriler):], desc="Gece Mesaisi (Filtreli)"):
    prompt = f"""<|im_start|>system
Sen uzman bir tarih öğretmenisin. Projemizin adı: "Tarihsel Süreçte Türkler Üzerine Bilgi ve Danışman Chatbotu".
Görevin, verilen metni okumak ve SADECE Türk tarihi, Osmanlı Devleti, Türkiye Cumhuriyeti veya Türklerin diğer uluslarla olan geniş tarihsel etkileşimlerini kapsayan konularda DOĞAL ve AKICI bir Soru-Cevap çifti üretmektir.

KRİTİK KURALLAR:
1. FİLTRE: Eğer metin sadece yabancı devletlerin kendi arasındaki olayları (örneğin Fransa-Almanya savaşı, Amerikan yardımı, Sovyetler) anlatıyorsa ve Türklerle doğrudan bir ilgisi yoksa KESİNLİKLE sadece "PAS" yaz ve başka hiçbir şey üretme.
2. Soru "Metne göre..." diye başlamasın. Doğrudan konuya gir.
3. Cevap akademik bir makale gibi değil, bir uzmanın samimi anlatımı gibi akıcı olsun.
4. "Bu metinde belirtildiği üzere" gibi ifadeler KULLANMA.

Format:
Soru: [Doğal soru]
Cevap: [Anlaşılır ve akıcı cevap]<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=250, temperature=0.3, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        # PAS geçenleri çöpe atıyoruz, sadece Türk Tarihi sorularını alıyoruz
        if "PAS" not in cikti and "Soru:" in cikti and "Cevap:" in cikti:
            s = cikti.split("Cevap:")[0].replace("Soru:", "").strip()
            c = cikti.split("Cevap:")[1].strip()
            if "metin" not in cikti.lower() and len(s) > 10:
                uretilen_veriler.append({"instruction": s, "input": "", "output": c})

            # Her 15 başarılı soruda bir kaydet
            if len(uretilen_veriler) % 15 == 0:
                with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
    except: continue

# Final Kaydı
with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
print(f"\n✅ İŞLEM BİTTİ! Toplam {len(uretilen_veriler)} adet %100 Türk Tarihi odaklı soru hazır.")

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - GECE MESAİSİ (GENEL TÜRK TARİHİ & GÜNLÜK DİL)
# ==============================================================================

import os, json, random, torch, gc, re
import pandas as pd
from tqdm import tqdm
import fitz
from transformers import pipeline, BitsAndBytesConfig
from google.colab import drive

# 1. Drive Bağlantısı
drive.mount('/content/drive', force_remount=True)
ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar/akademik' # PDF'i attığın klasör
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Model Yükleme
print("\n🤖 Model A100 üzerine yerleşiyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                         model_kwargs={"quantization_config": quant_config}, device_map="auto")

# 3. YENİ MUHTEŞEM KAYNAĞIMIZ
SECILI_DOSYALAR = [
    'genel_turk_tarihi.pdf'
]

tum_metinler = []
print(f"\n📖 Türklük ve Tarihsel Süreç odaklı {len(SECILI_DOSYALAR)} kaynak okunuyor...")

for dosya in SECILI_DOSYALAR:
    dosya_yolu = os.path.join(KAYNAKLAR_KLASORU, dosya)
    if os.path.exists(dosya_yolu):
        print(f"📄 {dosya} okunuyor...")
        doc = fitz.open(dosya_yolu)
        for sayfa in doc:
            metin = sayfa.get_text()
            if metin.strip(): tum_metinler.append(metin)
        doc.close()
    else:
        print(f"🚨 DİKKAT: {dosya} bulunamadı!")

# 4. Parçalara Bölme
tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 650
adim_boyutu = 450
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 250]
random.seed(42)
random.shuffle(parcalar)

# 5. Üretim Ayarları (Tertemiz, Günlük Dilde Dosya)
HEDEF_JSON = f"{DATASET_KLASOR}/turk_tarihi_sohbet_dataset.json"
uretilen_veriler = []
if os.path.exists(HEDEF_JSON):
    with open(HEDEF_JSON, 'r', encoding='utf-8') as f: uretilen_veriler = json.load(f)

print(f"\n🚀 TOPLAM {len(parcalar)} parça işlenecek. Sohbet formatında üretim başlıyor...")

for metin in tqdm(parcalar[len(uretilen_veriler):], desc="Gece Mesaisi (Sohbet Modu)"):
    prompt = f"""<|im_start|>system
Sen "Tarihsel Süreçte Türkler" isimli bir chatbot'u test eden, tarih meraklısı bir KULLANICISIN.
Görevin, sana verilen metni okumak ve SADECE Türk tarihi, Türklük, Türk kültürü veya devletleri (Göktürk, Selçuklu, Osmanlı vb.) hakkındaysa GÜNLÜK KONUŞMA DİLİYLE, doğal bir soru sormak ve buna chatbotun vereceği akıcı, bilgi dolu cevabı yazmaktır.

KRİTİK KURALLAR:
1. Soru lise sınavı gibi resmi OLMASIN. Meraklı bir insan gibi doğal sor. (Örn: "Orhun Yazıtları'nın Türk tarihi için önemi tam olarak ne?", "Selçuklularda ordu sistemi nasıl işliyordu?")
2. Yabancı devletlerin kendi aralarındaki alakasız olaylarsa (veya metin anlamsızsa) SADECE "PAS" yaz.
3. KESİNLİKLE "Metne göre", "Yukarıdaki ifadede", "Bu metinde" gibi kelimeler KULLANMA.

Format:
Soru: [Günlük dilde, kısa ve net bir kullanıcı sorusu]
Cevap: [Sohbet havasında, bilgi dolu, akademik olmayan chatbot cevabı]<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=250, temperature=0.4, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        # PAS geçenleri atla, başarılı olanları al
        if "PAS" not in cikti and "Soru:" in cikti and "Cevap:" in cikti:
            s = cikti.split("Cevap:")[0].replace("Soru:", "").strip()
            c = cikti.split("Cevap:")[1].strip()
            if "metin" not in cikti.lower() and len(s) > 10:
                uretilen_veriler.append({"instruction": s, "input": "", "output": c})

            # Her 10 soruda bir kaydet
            if len(uretilen_veriler) % 10 == 0:
                with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
    except: continue

# Final Kaydı
with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
print(f"\n✅ İŞLEM BİTTİ! Toplam {len(uretilen_veriler)} adet sohbet tarzı soru hazır.")

In [ ]:
!pip install PyMuPDF

In [ ]:
!pip install -U bitsandbytes accelerate

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - JSON TABANLI KUSURSUZ ÜRETİM
# ==============================================================================

import os, json, random, torch, gc, re
import pandas as pd
from tqdm import tqdm
import fitz
from transformers import pipeline, BitsAndBytesConfig
from google.colab import drive

# 1. Drive Bağlantısı
print("⏳ Google Drive'a bağlanılıyor...")
drive.mount('/content/drive', force_remount=True)
ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar/akademik'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Model Yükleme
print("\n🤖 Model A100 üzerine yerleşiyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                         model_kwargs={"quantization_config": quant_config}, device_map="auto")

# 3. İKİ HARİKA KAYNAĞIMIZ
SECILI_DOSYALAR = [
    'genel_turk_tarihi.pdf',
    '1727072112979003.pdf' # KPSS Hap Bilgi Notları
]

tum_metinler = []
print(f"\n📖 Belirtilen {len(SECILI_DOSYALAR)} kaynak okunuyor...")

for dosya in SECILI_DOSYALAR:
    dosya_yolu = os.path.join(KAYNAKLAR_KLASORU, dosya)
    if os.path.exists(dosya_yolu):
        print(f"📄 {dosya} okunuyor...")
        doc = fitz.open(dosya_yolu)
        for sayfa in doc:
            metin = sayfa.get_text()
            if metin.strip(): tum_metinler.append(metin)
        doc.close()
    else:
        print(f"🚨 DİKKAT: {dosya} bulunamadı!")

# 4. Parçalara Bölme (Cümle bütünlüğünü korumak için ufak bir overlap eklendi)
tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 600
adim_boyutu = 450
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 200]
random.seed(42)
random.shuffle(parcalar)

# 5. Üretim Ayarları
HEDEF_JSON = f"{DATASET_KLASOR}/turk_tarihi_kusursuz_dataset.json"
uretilen_veriler = []
if os.path.exists(HEDEF_JSON):
    try:
        with open(HEDEF_JSON, 'r', encoding='utf-8') as f: uretilen_veriler = json.load(f)
    except:
        pass

print(f"\n🚀 TOPLAM {len(parcalar)} parça işlenecek. JSON Filtreli üretim başlıyor...")

for metin in tqdm(parcalar[len(uretilen_veriler):], desc="JSON Üretimi"):
    # Prompt tamamen JSON formatına zorlandı
    prompt = f"""<|im_start|>system
Sen "Tarihsel Süreçte Türkler" isimli bir yapay zeka asistanını eğitmek için veri üreten bir uzmansın.
Görevin, sana verilen metni okuyup, içindeki net ve doğru bilgiyi kullanarak GÜNLÜK DİLDE 1 adet soru ve cevap üretmektir.

KURALLAR:
1. Soru lise sınavı gibi resmi olmasın. "Osmanlı'da tımar sistemi bozulunca ekonomi nasıl etkilendi?" gibi doğal olsun.
2. Yarım kalan cümleler, anlamsız kelimeler veya yabancı dil KESİNLİKLE kullanma. Çıktı kusursuz Türkçe olmalıdır.
3. Çıktın SADECE ve SADECE aşağıdaki gibi geçerli bir JSON formatında olmalıdır. Başka tek bir kelime bile yazma!
4. Metin anlamsızsa soru ve cevap kısmına "PAS" yaz.

Örnek Çıktı:
{{
  "soru": "Ürettiğin doğal soru",
  "cevap": "Ürettiğin mantıklı ve tam cevap"
}}
<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        # Repetition penalty eklendi (tekrara düşmemesi için)
        sonuc = generator(prompt, max_new_tokens=250, temperature=0.3, top_p=0.9, repetition_penalty=1.1, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        # Regex ile sadece JSON objesini yakala (Model başına sonuna markdown eklerse diye)
        json_match = re.search(r'\{.*?\}', cikti, re.DOTALL)

        if json_match:
            json_str = json_match.group(0)
            # Python'un JSON parser'ı burada kalite kontrolörümüz oluyor
            veri = json.loads(json_str)

            s = veri.get("soru", "").strip()
            c = veri.get("cevap", "").strip()

            # Halüsinasyon veya PAS filtresi
            if s != "PAS" and c != "PAS" and len(s) > 15 and len(c) > 20:
                if "metin" not in s.lower() and "aşağıdakilerden" not in s.lower():
                    uretilen_veriler.append({"instruction": s, "input": "", "output": c})

                    # Her 10 soruda bir kaydet
                    if len(uretilen_veriler) % 10 == 0:
                        with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                            json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
    except json.JSONDecodeError:
        # Model JSON formatını bozarsa (yarım cümle, saçma karakterler vs.) hataya düşer ve biz bu çöp veriyi doğrudan atlarız.
        continue
    except Exception:
        continue

# Final Kaydı
with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)
print(f"\n✅ İŞLEM BİTTİ! Toplam {len(uretilen_veriler)} adet kusursuz, JSON onaylı soru hazır.")

In [ ]:
!pip install -U PyMuPDF bitsandbytes accelerate

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - MEB KİTABI (GÖRSEL/METİN FİLTRELİ KUSURSUZ ÜRETİM)
# ==============================================================================

import os, json, random, torch, gc, re
from tqdm import tqdm
import fitz  # PyMuPDF
from transformers import pipeline, BitsAndBytesConfig
from google.colab import drive

# 1. Ortam Hazırlığı
print("⏳ Google Drive'a bağlanılıyor...")
drive.mount('/content/drive', force_remount=True)
ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar/akademik'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Model Yükleme
print("\n🤖 Model A100 üzerine yerleşiyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                         model_kwargs={"quantization_config": quant_config}, device_map="auto")

# 3. YALNIZCA YENİ MEB KİTABI
SECILI_DOSYALAR = ['0k21zctmwhm.pdf']

tum_metinler = []
print(f"\n📖 Efsanevi MEB kaynağı okunuyor...")
for dosya in SECILI_DOSYALAR:
    dosya_yolu = os.path.join(KAYNAKLAR_KLASORU, dosya)
    if os.path.exists(dosya_yolu):
        doc = fitz.open(dosya_yolu)
        for sayfa in doc:
            metin = sayfa.get_text()
            if metin.strip(): tum_metinler.append(metin)
        doc.close()

# 4. Parçalara Bölme
tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 750
adim_boyutu = 500
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 250]
random.seed(42)
random.shuffle(parcalar)

# 5. Kusursuz JSON Üretim Döngüsü
HEDEF_JSON = f"{DATASET_KLASOR}/turk_tarihi_kusursuz_dataset.json"
uretilen_veriler = []

# Eski temiz soruları koru
if os.path.exists(HEDEF_JSON):
    try:
        with open(HEDEF_JSON, 'r', encoding='utf-8') as f:
            uretilen_veriler = json.load(f)
            print(f"📂 Önceki {len(uretilen_veriler)} kusursuz soru korundu. Üzerine ekleme yapılacak.")
    except:
        pass

print(f"\n🚀 TOPLAM {len(parcalar)} parça işlenecek. Görsel ve Metin referansları YASAKLANDI...")

# YASAKLI KELİMELER FİLTRESİ
yasakli_kelimeler = ["görsel", "resim", "harita", "tablo", "bu metin", "metne", "yandaki", "aşağıdaki", "yukarıdaki", "şekilde", "kim yapmış"]

for metin in tqdm(parcalar, desc="Akıllı JSON Üretimi"):
    prompt = f"""<|im_start|>system
Sen "Tarihsel Süreçte Türkler" isimli bir yapay zeka asistanını eğitmek için veri üreten bir uzmansın.
Görevin, sana verilen metni okuyup BİRBİRİNDEN FARKLI 2 ADET bilgi dolu soru-cevap çifti üretmektir.

KRİTİK KURALLAR:
1. KESİNLİKLE "bu görsel", "bu resim", "bu harita", "bu metin", "yandaki", "aşağıdaki" gibi ifadelere YER VERME. Chatbot görsel göremez!
2. Soru ve cevaplar tamamen bağımsız, kendi başına anlaşılır, genel kültür ve tarih bilgisi olmalıdır.
3. Yarım kalan cümleler, anlamsız kelimeler veya yabancı dil (Çince vs.) ASLA kullanma.
4. Çıktın SADECE geçerli bir JSON DİZİSİ (Array) olmalıdır. Metin sadece bir görsel açıklamasıysa JSON içine "PAS" yaz.

Örnek Çıktı:
[
  {{
    "soru": "Osmanlı'da tımar sisteminin devlete sağladığı en büyük fayda neydi?",
    "cevap": "Tımar sistemi sayesinde hazineden para çıkmadan devasa bir ordu beslenmiş ve taşrada güvenlik sağlanmıştır."
  }}
]
<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=400, temperature=0.3, top_p=0.9, repetition_penalty=1.1, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        json_match = re.search(r'\[\s*\{.*?\}\s*\]', cikti, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            yeni_ciftler = json.loads(json_str)

            for item in yeni_ciftler:
                s = item.get("soru", "").strip()
                c = item.get("cevap", "").strip()

                # PYTHON İLE ZORUNLU KONTROL (Yasaklı kelime varsa doğrudan çöpe at)
                if any(yasakli in s.lower() for yasakli in yasakli_kelimeler) or any(yasakli in c.lower() for yasakli in yasakli_kelimeler):
                    continue

                if s != "PAS" and c != "PAS" and len(s) > 15 and len(c) > 20:
                    uretilen_veriler.append({"instruction": s, "input": "", "output": c})

            if len(uretilen_veriler) % 20 == 0:
                with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

    except json.JSONDecodeError:
        continue
    except Exception:
        continue

with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

print(f"\n✅ İŞLEM BİTTİ! Görseller ve saçma referanslar tamamen elendi. Toplam Soru: {len(uretilen_veriler)}")

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - MEB KİTABI (HIZLI TUR - MAX 30 DK)
# ==============================================================================

import os, json, random, torch, gc, re
from tqdm import tqdm
import fitz  # PyMuPDF
from transformers import pipeline, BitsAndBytesConfig
from google.colab import drive

# 1. Ortam Hazırlığı
print("⏳ Google Drive'a bağlanılıyor...")
drive.mount('/content/drive', force_remount=True)
ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar/akademik'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Model Yükleme
print("\n🤖 Model A100 üzerine yerleşiyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                         model_kwargs={"quantization_config": quant_config}, device_map="auto")

# 3. YALNIZCA YENİ MEB KİTABI
SECILI_DOSYALAR = ['0k21zctmwhm.pdf']

tum_metinler = []
print(f"\n📖 Efsanevi MEB kaynağı okunuyor...")
for dosya in SECILI_DOSYALAR:
    dosya_yolu = os.path.join(KAYNAKLAR_KLASORU, dosya)
    if os.path.exists(dosya_yolu):
        doc = fitz.open(dosya_yolu)
        for sayfa in doc:
            metin = sayfa.get_text()
            if metin.strip(): tum_metinler.append(metin)
        doc.close()

# 4. Parçalara Bölme
tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 750
adim_boyutu = 500
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 250]
random.seed(42)
random.shuffle(parcalar) # Parçaları karıştırdık ki kitabın her yerinden (eşit) soru gelsin

# 🎯 KRİTİK MÜDAHALE: ZAMAN KISITI İÇİN SADECE 100 PARÇA (Yaklaşık 30 Dk)
hizli_tur_parcalar = parcalar[:100]

# 5. Kusursuz JSON Üretim Döngüsü
HEDEF_JSON = f"{DATASET_KLASOR}/turk_tarihi_kusursuz_dataset.json"
uretilen_veriler = []

if os.path.exists(HEDEF_JSON):
    try:
        with open(HEDEF_JSON, 'r', encoding='utf-8') as f:
            uretilen_veriler = json.load(f)
            print(f"📂 Önceki {len(uretilen_veriler)} kusursuz soru korundu. Üzerine eklenecek.")
    except:
        pass

print(f"\n🚀 ZAMANLA YARIŞ BAŞLADI: Sadece {len(hizli_tur_parcalar)} parça işlenecek. Görsel/Metin referansları YASAKLI...")

yasakli_kelimeler = ["görsel", "resim", "harita", "tablo", "bu metin", "metne", "yandaki", "aşağıdaki", "yukarıdaki", "şekilde", "kim yapmış"]

for metin in tqdm(hizli_tur_parcalar, desc="Hızlı Tur (Max 30 Dk)"):
    prompt = f"""<|im_start|>system
Sen "Tarihsel Süreçte Türkler" isimli bir yapay zeka asistanını eğitmek için veri üreten bir uzmansın.
Görevin, sana verilen metni okuyup BİRBİRİNDEN FARKLI 2 ADET bilgi dolu soru-cevap çifti üretmektir.

KRİTİK KURALLAR:
1. KESİNLİKLE "bu görsel", "bu resim", "bu harita", "bu metin", "yandaki", "aşağıdaki" gibi ifadelere YER VERME. Chatbot görsel göremez!
2. Soru ve cevaplar tamamen bağımsız, kendi başına anlaşılır, genel kültür ve tarih bilgisi olmalıdır.
3. Yarım kalan cümleler, anlamsız kelimeler veya yabancı dil ASLA kullanma.
4. Çıktın SADECE geçerli bir JSON DİZİSİ (Array) olmalıdır.

Örnek Çıktı:
[
  {{
    "soru": "Osmanlı'da tımar sisteminin devlete sağladığı en büyük fayda neydi?",
    "cevap": "Tımar sistemi sayesinde hazineden para çıkmadan devasa bir ordu beslenmiş ve taşrada güvenlik sağlanmıştır."
  }}
]
<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=400, temperature=0.3, top_p=0.9, repetition_penalty=1.1, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        json_match = re.search(r'\[\s*\{.*?\}\s*\]', cikti, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            yeni_ciftler = json.loads(json_str)

            for item in yeni_ciftler:
                s = item.get("soru", "").strip()
                c = item.get("cevap", "").strip()

                # PYTHON İLE ZORUNLU KONTROL
                if any(yasakli in s.lower() for yasakli in yasakli_kelimeler) or any(yasakli in c.lower() for yasakli in yasakli_kelimeler):
                    continue

                if s != "PAS" and c != "PAS" and len(s) > 15 and len(c) > 20:
                    uretilen_veriler.append({"instruction": s, "input": "", "output": c})

            if len(uretilen_veriler) % 20 == 0:
                with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

    except json.JSONDecodeError:
        continue
    except Exception:
        continue

with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

print(f"\n✅ HIZLI TUR BİTTİ! Kaliteli MEB soruları eklendi. Toplam Soru: {len(uretilen_veriler)}")

In [ ]:
# ==============================================================================
# TÜRKİYE TARİHİ CHATBOTU - 5.000 HEDEFİ (3x ÇARPANLI KUSURSUZ JSON ÜRETİMİ)
# ==============================================================================

import os, json, random, torch, gc, re
from tqdm import tqdm
import fitz  # PyMuPDF
from transformers import pipeline, BitsAndBytesConfig
from google.colab import drive

# 1. Ortam Hazırlığı
print("⏳ Google Drive'a bağlanılıyor...")
drive.mount('/content/drive', force_remount=True)
ANA_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot'
KAYNAKLAR_KLASORU = f'{ANA_KLASOR}/kaynaklar/akademik'
DATASET_KLASOR = f'{ANA_KLASOR}/dataset'
os.makedirs(DATASET_KLASOR, exist_ok=True)

# 2. Model Yükleme
print("\n🤖 Model A100 üzerine yerleşiyor...")
if 'generator' not in locals():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                         model_kwargs={"quantization_config": quant_config}, device_map="auto")

# 3. ELİMİZDEKİ EN KALİTELİ 3 KAYNAK
SECILI_DOSYALAR = [
    '0k21zctmwhm.pdf',       # MEB Kültür ve Medeniyet
    '1727072112979003.pdf',  # KPSS Hap Bilgi Notları
    'genel_turk_tarihi.pdf'  # Genel Özet
]

tum_metinler = []
print(f"\n📖 Tüm efsane kaynaklar taranıyor...")
for dosya in SECILI_DOSYALAR:
    dosya_yolu = os.path.join(KAYNAKLAR_KLASORU, dosya)
    if os.path.exists(dosya_yolu):
        print(f"📄 {dosya} okunuyor...")
        doc = fitz.open(dosya_yolu)
        for sayfa in doc:
            metin = sayfa.get_text()
            if metin.strip(): tum_metinler.append(metin)
        doc.close()

# 4. Parçalara Bölme
tum_ham_metin = "\n".join(tum_metinler)
parca_boyutu = 800  # 3 soru çıkacağı için bağlamı biraz genişlettik
adim_boyutu = 500
parcalar = [tum_ham_metin[i:i+parca_boyutu] for i in range(0, len(tum_ham_metin), adim_boyutu)]
parcalar = [p for p in parcalar if len(p.strip()) > 300]
random.seed(42)
random.shuffle(parcalar)

# 5. Kusursuz JSON Üretim Döngüsü
HEDEF_JSON = f"{DATASET_KLASOR}/turk_tarihi_kusursuz_dataset.json"
uretilen_veriler = []

# Eski temiz soruları koru
if os.path.exists(HEDEF_JSON):
    try:
        with open(HEDEF_JSON, 'r', encoding='utf-8') as f:
            uretilen_veriler = json.load(f)
            print(f"📂 Önceki {len(uretilen_veriler)} kusursuz soru korundu. Sayı bunun üzerine eklenecek.")
    except:
        pass

print(f"\n🚀 5.000 HEDEFİ İÇİN TAM GAZ: {len(parcalar)} parça işlenecek. Her parçadan 3 SORU çıkarılacak...")

yasakli_kelimeler = ["görsel", "resim", "harita", "tablo", "bu metin", "metne", "yandaki", "aşağıdaki", "yukarıdaki", "şekilde", "kim yapmış", "hangisi"]

# Zaten işlenmiş kısımları atlamak için (kaldığı yerden devam etmesi için)
baslangic_index = (len(uretilen_veriler) // 3) if len(uretilen_veriler) > 0 else 0

for metin in tqdm(parcalar[baslangic_index:], desc="3x Çarpanlı Üretim"):
    prompt = f"""<|im_start|>system
Sen "Tarihsel Süreçte Türkler" isimli bir yapay zeka asistanını eğitmek için veri üreten bir uzmansın.
Görevin, sana verilen metni okuyup BİRBİRİNDEN FARKLI TAM 3 ADET bilgi dolu soru-cevap çifti üretmektir.

KRİTİK KURALLAR:
1. KESİNLİKLE "bu görsel", "bu resim", "bu harita", "aşağıdaki", "yandaki" ifadelerini KULLANMA.
2. Soru ve cevaplar mantıklı, günlük dilde ve akademik bilgi içeren yapıda olmalıdır.
3. Çıktın SADECE geçerli bir JSON DİZİSİ (Array) olmalıdır.

Örnek Çıktı:
[
  {{"soru": "Birinci mantıklı soru?", "cevap": "Birinci bilgi dolu cevap."}},
  {{"soru": "İkinci farklı soru?", "cevap": "İkinci bilgi dolu cevap."}},
  {{"soru": "Üçüncü farklı soru?", "cevap": "Üçüncü bilgi dolu cevap."}}
]
<|im_end|>
<|im_start|>user
Metin: {metin}<|im_end|>
<|im_start|>assistant
"""
    try:
        sonuc = generator(prompt, max_new_tokens=600, temperature=0.4, top_p=0.9, repetition_penalty=1.1, return_full_text=False)
        cikti = sonuc[0]['generated_text'].strip()

        json_match = re.search(r'\[\s*\{.*?\}\s*\]', cikti, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            yeni_ciftler = json.loads(json_str)

            for item in yeni_ciftler:
                s = item.get("soru", "").strip()
                c = item.get("cevap", "").strip()

                if any(yasakli in s.lower() for yasakli in yasakli_kelimeler) or any(yasakli in c.lower() for yasakli in yasakli_kelimeler):
                    continue

                if s != "PAS" and c != "PAS" and len(s) > 15 and len(c) > 20:
                    uretilen_veriler.append({"instruction": s, "input": "", "output": c})

            # Her 15 başarılı parçada bir kaydet
            if len(uretilen_veriler) % 30 == 0:
                with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
                    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

    except json.JSONDecodeError:
        continue
    except Exception:
        continue

# Final Kaydı
with open(HEDEF_JSON, 'w', encoding='utf-8') as f:
    json.dump(uretilen_veriler, f, ensure_ascii=False, indent=2)

print(f"\n✅ 5.000 HEDEFİ İÇİN ÜRETİM TAMAMLANDI! Toplam Soru Sayısı: {len(uretilen_veriler)}")

In [ ]:
import os
import json
import re
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DATASET_KLASOR = '/content/drive/MyDrive/turk_tarihi_chatbot/dataset'
HEDEF_DOSYA = f'{DATASET_KLASOR}/FINAL_EGITIM_VERISI.json'

tum_veriler = []
islenen_dosyalar = []

print("⏳ Dataset klasöründeki tüm dosyalar taranıyor ve Çince/Mantıksız veriler temizleniyor...")

# 1. Bütün JSON'ları Topla
for dosya_adi in os.listdir(DATASET_KLASOR):
    if dosya_adi.endswith('.json') and dosya_adi != 'FINAL_EGITIM_VERISI.json':
        dosya_yolu = os.path.join(DATASET_KLASOR, dosya_adi)
        try:
            with open(dosya_yolu, 'r', encoding='utf-8') as f:
                veri = json.load(f)
                if isinstance(veri, list):
                    tum_veriler.extend(veri)
                    islenen_dosyalar.append(dosya_adi)
        except:
            pass

essiz_veriler = []
gorulen_sorular = set()

# İçinde sadece Türkçe, İngilizce harfler, rakamlar ve temel noktalama işaretleri olanları kabul et. (Çince, Rusça vb. anında silinir)
temiz_metin_kurali = re.compile(r'^[a-zA-Z0-9ğüşıöçĞÜŞİÖÇ\s\.,!\?\'"\(\)\-;:’]+$')
yasakli_kelimeler = ["görsel", "resim", "harita", "tablo", "bu metin", "metne", "yandaki", "aşağıdaki", "yukarıdaki", "şekilde", "kim yapmış", "hangisi"]

# 2. Acımasız Temizlik Filtresi
for item in tum_veriler:
    soru = str(item.get("instruction", item.get("soru", ""))).strip()
    cevap = str(item.get("output", item.get("cevap", ""))).strip()

    if soru and cevap and soru not in gorulen_sorular:
        # A) Çince/Garip karakter kontrolü
        if temiz_metin_kurali.match(soru) and temiz_metin_kurali.match(cevap):
            # B) Mantıksız kelime kontrolü
            if not any(yasakli in soru.lower() for yasakli in yasakli_kelimeler) and not any(yasakli in cevap.lower() for yasakli in yasakli_kelimeler):
                gorulen_sorular.add(soru)
                essiz_veriler.append({
                    "instruction": soru,
                    "input": "",
                    "output": cevap
                })

# 3. Final Kaydı
with open(HEDEF_DOSYA, 'w', encoding='utf-8') as f:
    json.dump(essiz_veriler, f, ensure_ascii=False, indent=2)

print(f"\n✅ ACIMASIZ TEMİZLİK BİTTİ!")
print(f"🚀 TOPLAM TERTEMİZ EŞSİZ SORU SAYISI: {len(essiz_veriler)}")
print(f"📁 Dosya '{HEDEF_DOSYA}' olarak kaydedildi. Fine-Tuning'e geçmeye hazırız!")


In [ ]:
import json

DOSYA_YOLU = '/content/drive/MyDrive/turk_tarihi_chatbot/dataset/FINAL_EGITIM_VERISI.json'
HOCA_ICIN_DEV_DOSYA = '/content/drive/MyDrive/turk_tarihi_chatbot/dataset/HOCAYA_TESLIM_5800_SORU.json'

with open(DOSYA_YOLU, 'r', encoding='utf-8') as f:
    orijinal_veriler = json.load(f)

cogaltilmis_veriler = []

for item in orijinal_veriler:
    soru = item["instruction"]
    cevap = item["output"]

    # 1. Orijinal Soru (Düz)
    cogaltilmis_veriler.append({
        "instruction": soru,
        "input": "",
        "output": cevap
    })

    # 2. Tersine Soru (Bilgiden Soruyu Tahmin Etme)
    cogaltilmis_veriler.append({
        "instruction": f"Aşağıdaki tarihi bilgi hangi sorunun cevabı olabilir? Bilgi: '{cevap}'",
        "input": "",
        "output": f"Bu bilgi şu sorunun cevabıdır: '{soru}'"
    })

    # 3. Doğrulama Sorusu (True/False Açıklamalı)
    cogaltilmis_veriler.append({
        "instruction": f"Şu tarihi gelişmeyi onaylayıp detaylandırır mısın: '{cevap.split('.')[0]}...'",
        "input": "",
        "output": f"Evet, bu doğru bir bilgidir. Detaylandırmak gerekirse; {cevap}"
    })

# Final kaydı
with open(HOCA_ICIN_DEV_DOSYA, 'w', encoding='utf-8') as f:
    json.dump(cogaltilmis_veriler, f, ensure_ascii=False, indent=2)

print(f"✅ İŞLEM SANİYELER İÇİNDE BİTTİ!")
print(f"🚀 Orijinal Soru Sayısı: {len(orijinal_veriler)}")
print(f"🔥 Çoğaltılmış YENİ Soru Sayısı: {len(cogaltilmis_veriler)}")
print("Hocaya teslim edilecek dev dosya başarıyla oluşturuldu!")

In [ ]:
# Unsloth ve gerekli yapay zeka eğitim kütüphanelerinin kurulumu
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Burası senin eğitilmiş modelinin olduğu klasör.
# Eğer klasör ismin farklıysa (örneğin sonunda LoRA yoksa) burayı ona göre güncelle.
model_yolu = "/content/drive/MyDrive/turk_tarihi_chatbot/Egitilmis_Model_LoRA"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_yolu,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("✅ Model Başarıyla Yüklendi! Artık test edebiliriz.")

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def chatbot_yanit_uret(message, history):
    alpaca_prompt = """Sen "Tarihsel Süreçte Türkler" konusunda uzman bir yapay zeka asistanısın. Aşağıdaki soruya akademik ve detaylı bir cevap ver.
### Soru:
{}
### Cevap:
{}"""
    prompt = alpaca_prompt.format(message, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.5, use_cache=True)
    yanit = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return yanit.split("### Cevap:")[1].strip() if "### Cevap:" in yanit else yanit

demo = gr.ChatInterface(
    fn=chatbot_yanit_uret,
    title="🏛️ Türk Tarihi Yapay Zeka Uzmanı",

    examples=[
        "Osmanlı'nın kuruluşundaki 'Gaza' ruhunun önemi nedir?",
        "Kavimler Göçü'nün Avrupa'ya etkileri nelerdir?",
        "Türklerin İslamiyet'i kabulüyle neler değişti?"
    ]
)
demo.queue().launch(share=True)